In [1]:
!pip install transformers==4.49.0 sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 39.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 116.2 MB/s  0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully uninstalled huggingface_hub-1.33.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.3
    Uninstalling tokenizers-0.23.3:
      Successfully uninstalled tokenizers-0.23.3
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.1
    Uninstalling transformers-5.22.1:
      Successfully uninstalled transformers-5.22.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [transformers]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency con

In [1]:
from transformers import pipeline

summarizer = pipeline(
    "summarization",
    model="sshleifer/distilbart-cnn-12-6"
)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Device set to use cpu


In [2]:
text = """
Artificial Intelligence is a technology that enables machines
to perform tasks that normally require human intelligence.
It is used in healthcare, education, banking and transportation.
AI helps automate tasks, analyze data and make predictions.
Machine learning is a part of AI that allows systems to learn
from data and improve their performance.
"""

summary = summarizer(
    text,
    max_length=60,
    min_length=20,
    do_sample=False
)

print("Original Text:")
print(text)

print("\nGenerated Summary:")
print(summary[0]["summary_text"])

Original Text:

Artificial Intelligence is a technology that enables machines
to perform tasks that normally require human intelligence.
It is used in healthcare, education, banking and transportation.
AI helps automate tasks, analyze data and make predictions.
Machine learning is a part of AI that allows systems to learn
from data and improve their performance.


Generated Summary:
 Artificial Intelligence is a technology that enables machines to perform tasks that normally require human intelligence . It is used in healthcare, education, banking and transportation .


In [3]:
def generate_summary(text, length):
    if length == "short":
        max_len = 40
        min_len = 10

    elif length == "medium":
        max_len = 80
        min_len = 30

    elif length == "long":
        max_len = 120
        min_len = 50

    else:
        return "Invalid length! Choose short, medium, or long."

    result = summarizer(
        text,
        max_length=max_len,
        min_length=min_len,
        do_sample=False
    )

    return result[0]["summary_text"]

In [4]:
print("SHORT SUMMARY:")
print(generate_summary(text, "short"))

print("\nMEDIUM SUMMARY:")
print(generate_summary(text, "medium"))

print("\nLONG SUMMARY:")
print(generate_summary(text, "long"))

SHORT SUMMARY:


Your max_length is set to 80, but your input_length is only 70. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=35)


 Artificial Intelligence enables machines to perform tasks that normally require human intelligence . It is used in healthcare, education, banking and transportation .

MEDIUM SUMMARY:


Your max_length is set to 120, but your input_length is only 70. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=35)


 Artificial Intelligence is a technology that enables machines to perform tasks that normally require human intelligence . It is used in healthcare, education, banking and transportation . AI helps automate tasks, analyze data and make predictions .

LONG SUMMARY:
 Artificial Intelligence is a technology that enables machines to perform tasks that normally require human intelligence . It is used in healthcare, education, banking and transportation . AI helps automate tasks, analyze data and make predictions . Machine learning is a part of AI that allows systems to learn from data .


In [6]:
def safe_summarize(text, length="medium"):

    # Check empty input
    if not text.strip():
        return "Please enter some text to summarize."

    # Handle very long input
    inputs = summarizer.tokenizer(
        text,
        max_length=900,
        truncation=True,
        return_tensors="pt"
    )

    text = summarizer.tokenizer.decode(
        inputs["input_ids"][0],
        skip_special_tokens=True
    )

    # Generate summary
    return generate_summary(text, length)

In [7]:
# Test 1: Empty input
print("TEST 1: Empty Input")
print(safe_summarize(""))

# Test 2: Valid input
print("\nTEST 2: Valid Input")
print(safe_summarize(text, "short"))

# Test 3: Very long input
long_text = text * 100

print("\nTEST 3: Long Input")
print(safe_summarize(long_text, "medium"))

TEST 1: Empty Input
Please enter some text to summarize.

TEST 2: Valid Input
 Artificial Intelligence enables machines to perform tasks that normally require human intelligence . It is used in healthcare, education, banking and transportation .

TEST 3: Long Input
 Artificial Intelligence is a technology that enables machines to perform tasks that normally require human intelligence . It is used in healthcare, education, banking and transportation . It helps automate tasks, analyze data and make predictions .


In [8]:
test_examples = [
    "Artificial Intelligence helps machines perform tasks that normally require human intelligence.",

    "Education is important because it provides knowledge, develops skills and helps people build better careers.",

    "Climate change causes rising temperatures, melting glaciers and changes in weather patterns.",

    "The internet allows people to communicate, share information and access educational resources.",

    "Renewable energy sources such as solar and wind power help reduce pollution and protect the environment."
]

for i, example in enumerate(test_examples, 1):
    print(f"\nExample {i}")
    print("Input:", example)
    print("Summary:", safe_summarize(example, "short"))

Your max_length is set to 40, but your input_length is only 14. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=7)



Example 1
Input: Artificial Intelligence helps machines perform tasks that normally require human intelligence.


Your max_length is set to 40, but your input_length is only 19. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=9)


Summary:  Artificial Intelligence helps machines perform tasks that normally require human intelligence .

Example 2
Input: Education is important because it provides knowledge, develops skills and helps people build better careers.


Your max_length is set to 40, but your input_length is only 16. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=8)


Summary:  Education provides knowledge, develops skills and helps people build better careers . Education is important because it provides knowledge and develops skills .

Example 3
Input: Climate change causes rising temperatures, melting glaciers and changes in weather patterns.


Your max_length is set to 40, but your input_length is only 16. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=8)


Summary:  Climate change causes rising temperatures, melting glaciers and changes in weather patterns .

Example 4
Input: The internet allows people to communicate, share information and access educational resources.


Your max_length is set to 40, but your input_length is only 20. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=10)


Summary:  The internet allows people to communicate, share information and access educational resources .

Example 5
Input: Renewable energy sources such as solar and wind power help reduce pollution and protect the environment.
Summary:  Renewable energy sources such as solar and wind power help reduce pollution and protect the environment .


In [9]:
import pandas as pd

evaluation = {
    "Example": [1, 2, 3, 4, 5],
    "Main Idea Preserved": [
        "Yes", "Yes", "Yes", "Yes", "Yes"
    ],
    "Factual Accuracy": [
        "Correct", "Correct", "Correct", "Correct", "Correct"
    ],
    "Remarks": [
        "Main idea retained",
        "Important points retained",
        "Key information retained",
        "Meaning preserved",
        "Main concept retained"
    ]
}

df = pd.DataFrame(evaluation)
display(df)

,Example,Main Idea Preserved,Factual Accuracy,Remarks
0,1,Yes,Correct,Main idea retained
1,2,Yes,Correct,Important points retained
2,3,Yes,Correct,Key information retained
3,4,Yes,Correct,Meaning preserved
4,5,Yes,Correct,Main concept retained


# AI Text Summarizer

## 1. Project Description
The AI Text Summarizer is a Python-based application that uses a pre-trained language model to convert long text into shorter summaries while preserving the main meaning.

## 2. Model Used
- Model: DistilBART CNN
- Model ID: sshleifer/distilbart-cnn-12-6
- Library: Hugging Face Transformers
- Programming Language: Python

## 3. Features
- Accepts text input from users.
- Generates short, medium, and long summaries.
- Handles empty input.
- Handles excessively long input through truncation.
- Evaluates five different text examples.

## 4. Prompt and Model Usage
The summarization model receives the input text and generates a shorter version while attempting to retain the important information.

## 5. Evaluation
Five different examples are tested to check factual accuracy and whether the main idea is preserved.